# 15장. Multi-agent로 매출 분석을 분담하고 보고서 통합하기

CSV와 목표 JSON을 두 담당이 분석하고 메인이 보고서로 통합합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 하위 에이전트 위임·담당별 이력·공유 결과 통합 |
| 이번 실습 조합 | OpenAI-hosted + 목표 JSON 직접 읽기 + 담당 두 명 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

선행 학습: 3장 세션·Turn, 5장 파일·artifact. 9장 Skills·11장 승인은 선택 확장입니다. Docker·executor 키 없이 OpenAI-hosted 환경에서 실행합니다.

1. `agent.multi_agent`로 위임을 활성화합니다.
2. 매출 구조와 목표 달성 분석을 서로 다른 하위 에이전트에게 맡깁니다.
3. 메인이 두 파일을 읽고 통합 보고서를 작성합니다.
4. 하위 에이전트의 생성 이벤트·저장된 명령·Turn을 확인합니다.
5. 파일을 내려받고 세션을 정리합니다.

하위 에이전트는 독립 문맥을 가지며 환경의 파일시스템은 공유합니다. 생성·메시지·대기·중단 도구는 Harness가 제공합니다. Python에서 별도 세션 두 개를 만드는 방식이나 Agents SDK의 handoff를 구현하는 실습이 아닙니다.

### 단일 에이전트와 비교하기

| 방식 | 이 예제에서의 동작 | 선택 기준 |
| --- | --- | --- |
| 단일 에이전트 | 매출·목표 분석을 순서대로 수행 | 짧거나 서로 의존하는 단계 |
| Multi-agent | 두 분석을 독립적으로 수행한 뒤 메인이 통합 | 분리 가능한 분석·검토 작업 |

작은 CSV는 한 에이전트로도 충분합니다. 여기서는 속도·비용 개선을 측정하기보다 위임 설정, 담당별 실행 이력, 공유 파일의 통합을 학습합니다. 작업 분리와 명확한 산출물 지정은 [공식 문서](https://developers.openai.com/api/docs/guides/agents-api/multi-agent)의 권장 방식입니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실행 환경 준비

로컬은 `uv sync --locked` 후 `.venv` 커널을 사용하세요. Colab에서는 독립 실행본을 업로드합니다. 실제 API에는 기존 프로젝트의 API 키·모델 설정과 `RUN_LIVE=True`가 필요합니다.

In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")

In [ ]:
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 15
CHAPTER = "15_multi_agent"
PERIOD = "2026-01"
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
csv_bytes = (settings.root / "data/sales.csv").read_bytes()
targets_bytes = (settings.root / "data/sales_targets.json").read_bytes()
print({"chapter": NUMBER, "live_enabled": settings.enabled, "model": settings.model or "Offline mode"})


## 1. 역할과 출력 파일 분리

| 담당 | 업무 | 전용 산출물 |
| --- | --- | --- |
| 매출 구조 분석 | 상품별·일별 매출, 행 수·수량 계산 | sales_breakdown.json |
| 목표 달성 분석 | 선택 기간의 목표와 실적 비교, 달성률·부족액 계산 | target_assessment.json |
| 메인 | 두 결과를 읽어 통합 | summary.json, report.md |

두 담당은 각자 입력을 읽고 자신의 파일만 씁니다. 메인은 담당 작업을 대신 수행하지 않습니다. 입력은 수정하지 않고 출력 파일명과 실행 스크립트를 분리합니다. 작은 CSV에서는 속도 향상을 보장하지 않으며 역할 분담과 API 사용법을 학습합니다.


In [ ]:
import base64
import json
import shlex
from pathlib import Path
from IPython.display import Markdown, display
import time
from uuid import uuid4
from tutorial_helpers.common import new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error

COORDINATOR_INSTRUCTIONS = (
    "You coordinate a sales analysis. Create exactly two direct subagents, one per assigned role. "
    "Tell them not to delegate further. Both must execute Python calculations independently. "
    "Do not perform either worker's task yourself. Give each a distinct script and output path. "
    "Wait for both completed results before combining their files using Python. "
    "Never browse the web, change inputs, or use external services. Keep commentary brief. "
    "Return only the summary.json JSON object as your final answer."
)
TASK = """Use the provided /workspace/sales.csv and /workspace/sales_targets.json.
Create exactly two subagents and send these separate tasks:

1. Sales structure: write and run python /workspace/workers/structure.py.
Read sales.csv with Python csv. Write /workspace/outputs/sales_breakdown.json as
{"metrics": {"row_count": int, "total_quantity": int, "total_revenue": int,
"by_product": {product: revenue}}, "by_date": {date: revenue}}.
Revenue is quantity * unit_price. Work only on this role; do not delegate.

2. Target assessment: write and run python /workspace/workers/target.py.
Independently calculate total revenue from sales.csv and read period 2026-01 in targets JSON.
Write /workspace/outputs/target_assessment.json as {"period": "2026-01",
"total_revenue": int, "target_revenue": int, "attainment_rate": float,
"shortfall": int, "currency": "KRW"}. attainment_rate=revenue/target. shortfall=max(target-revenue, 0).
Work only on this role; do not delegate. Do not read the structure worker's result.

Run each worker script in a separate command after writing it.
Both roles must read back their own output file, then report completion.
After both finish, use Python to read both JSON files and check their revenues agree.
Run the coordinator integration as a separate command beginning with python.
Write /workspace/outputs/summary.json as exactly the structure result's metrics object.
Write /workspace/outputs/report.md in Korean starting '# 매출 분석 보고서'.
Include total revenue, every product's revenue, daily revenue, target, and shortfall.
Display attainment_rate * 100 as a percentage with two decimal places, computed from the target assessment.
Explain both roles and cite the two input result files.
Read the four files back. Return exactly summary.json's object, without Markdown fences.
""".replace("2026-01", PERIOD)


### 공유 환경 요청 준비

두 입력 파일은 실제 바이트로 전송합니다. 출력된 목표 금액·통화는 계산 정답이 아닌 입력 데이터입니다. PERIOD를 변경하면 프롬프트의 분석 기간도 함께 바뀝니다.


In [ ]:
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [
        {"type": "inline", "path": "/workspace/sales.csv",
         "data": base64.b64encode(csv_bytes).decode()},
        {"type": "inline", "path": "/workspace/sales_targets.json",
         "data": base64.b64encode(targets_bytes).decode()},
    ],
}
print({"inputs": [f["path"] for f in environment["files"]], "max_concurrent_subagents": 2})
print("Selected target input:", {PERIOD: json.loads(targets_bytes)[PERIOD]})


## 2. Multi-agent 활성화와 스트리밍

`enabled=True`는 위임 도구를 활성화합니다. `max_concurrent_subagents=2`는 메인을 제외한 동시 실행 상한이며 두 작업이 반드시 동시에 실행된다는 보장은 아닙니다. 정확히 두 담당을 만들도록 입력에서 요청하고 실제 생성·실행 이력으로 확인합니다.

세션 이벤트에서는 두 하위 에이전트의 ID만 간단히 출력합니다. 전체 이벤트를 출력하거나 사용자 정의 Trace를 수집하지 않습니다. 생성·대기 호출 완료는 담당의 분석 완료와 다르므로 저장된 Turn과 파일도 확인합니다.

API 설정은 위임 도구와 동시 실행 상한을 정하고, 프롬프트는 두 담당의 업무를 요청하며, 실제 이력은 수행 결과를 보여줍니다. 전용 출력 경로는 작업 분담 규칙이며 접근 권한 격리가 아닙니다.


### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 셀부터 실행하세요. 이후 API 셀에는 실행 허용 분기를 반복하지 않습니다.

In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError("Set RUN_LIVE=True to run the API example. Use the validation script for offline checks.")
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)

## 3. 위임 실행부터 결과 통합 확인까지

| 단계 | 할 일 | 확인할 출력 |
| --- | --- | --- |
| 1. 위임 실행 | 실행 셀이 끝날 때까지 기다림 | Session·하위 에이전트 생성 ID |
| 2. 담당별 상태 조회 | 메인과 하위 Turn 비교 | 원래 root Turn과 담당별 상태 |
| 3. 역할별 명령 확인 | 역할·담당·Turn 연결 | 스크립트·실행 주체·종료 코드 |
| 4. 네 파일 다운로드 | 게시 이력과 저장 경로 확인 | Artifact ID·게시 Turn·원격/로컬 경로 |
| 5. 담당 결과와 보고서 읽기 | 두 분석 결과와 메인 통합 비교 | JSON 세 개와 보고서 본문 |
| 6. 정리 | 중간에 멈춰도 마지막 셀 실행 | Session 삭제·부재 확인 |

아래 상태 준비 셀은 새 client로 한 번만 실행합니다. 진행 중 다시 초기화하면 기존 소유권과 ID를 잃습니다. 각 단계 오류는 정리를 시도하므로 오류 뒤에는 기존 실행을 이어가지 말고 정리 상태를 확인한 뒤 준비부터 새로 시작하세요. 셀을 정상 종료하면 Session은 다음 단계까지 유지됩니다. 강제 커널 종료의 정리는 보장하지 않습니다.

### 핵심 API와 조회 범위

| API | 확인하는 내용 |
| --- | --- |
| `sessions.create` + `agent.multi_agent` | 세션을 만들고 Harness의 위임 도구 활성화 |
| `sessions.subagents.list` | 이 세션에서 생성한 하위 에이전트 목록 |
| `subagents.turns.list` | 담당별 Turn의 완료·실패 상태 |
| `subagents.items.list` | 담당별 명령·실행 이력 |
| `subagents.turns.retrieve` | 하위 Turn 조회; session_id·subagent_id·turn_id 지정 |
| `sessions.turns.retrieve` | 메인 Turn 조회; 메인의 subagent_id는 null |
| `sessions.artifacts.list` / `.content` | 공유 환경에서 게시된 결과 조회·다운로드 |

각 list 응답은 SDK 반복으로 모든 페이지를 읽습니다. limit는 전체 결과 상한이 아닌 페이지 크기입니다. 상세 실행 증거와 정답 검사는 별도 실행 사본에서 수행하며, 기본 실습에 custom Trace는 추가하지 않습니다.


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
created_subagent_ids = []
result = turn = None
session_creation_attempted = False
history_checked = commands_checked = report_downloaded = False
subagents, root_items, published, downloaded_artifacts = [], [], [], []
child_items, child_turns, command_actors, role_executions = {}, {}, {}, {}


### 1. 위임 실행

이 셀에서만 새 Session과 첫 입력을 생성합니다. 생성된 하위 ID는 역할 이름이 아니며 이벤트 순서도 역할 순서를 뜻하지 않습니다. 3단계에서 실제 스크립트로 역할을 연결합니다. 실행 중 출력이 잠시 없어도 담당 작업 중일 수 있습니다.

900초 예산은 이 셀에서 시작하며 이벤트 수신 시 검사합니다. HTTP timeout=90과 함께 사용하지만 엄격한 전체 실행 시간 제한은 아닙니다. timeout을 원격 작업 실패로 단정하지 않으며 이 학습 예제는 오류 시 정리를 시도합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if session_creation_attempted:
        raise RuntimeError("Session creation was already attempted; do not submit the work twice")
    session_creation_attempted = True
    started = time.monotonic()
    print("[1/6] Running the coordinator and its workers...", flush=True)
    with client.beta.agents.sessions.create(
        agent={
            "model": settings.model,
            "instructions": COORDINATOR_INSTRUCTIONS,
            "multi_agent": {"enabled": True, "max_concurrent_subagents": 2},
        },
        environment=environment,
        input=TASK,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.subagent.created":
                if event.subagent.id not in created_subagent_ids:
                    created_subagent_ids.append(event.subagent.id)
                    print("Subagent:", event.subagent.id)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action")
            else:
                check_event(event)
            if time.monotonic() - started > 900:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})


### 2. 메인과 담당별 실행 상태 조회

메인 Turn은 sessions.turns, 하위 Turn은 subagents.turns에서 조회합니다. 메인이 completed여도 모든 담당이 성공했는지 별도 확인해야 합니다. 생성 이벤트는 목록과 대조할 단서이며 담당 분석 완료 증거가 아닙니다. 아래에서는 담당별 Turn ID·상태·소유자를 출력하고 실패·취소 Turn을 성공으로 취급하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    history_checked = False
    if result is None:
        raise RuntimeError("Complete the delegation run in step 1 first")
    print("[2/6] Retrieving the root and worker histories...")
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15,
    )
    if turn.id != result.turn_id or turn.status != "completed" or turn.subagent_id is not None:
        raise RuntimeError("The original completed root turn was not confirmed")
    subagents = list(client.beta.agents.sessions.subagents.list(result.session_id, limit=100, timeout=15))
    child_items, child_turns = {}, {}
    print("Root turn:", turn.id, turn.status, "Observed child IDs:", created_subagent_ids)
    if len(subagents) != 2:
        raise RuntimeError("Expected two saved workers; inspect the delegation history")
    for subagent in subagents:
        child_items[subagent.id] = list(client.beta.agents.sessions.subagents.items.list(
            subagent.id, session_id=result.session_id, order="asc", limit=100, timeout=15,
        ))
        child_turns[subagent.id] = list(client.beta.agents.sessions.subagents.turns.list(
            subagent.id, session_id=result.session_id, order="asc", limit=100, timeout=15,
        ))
        print("Worker:", subagent.id, [(t.id, t.status, t.subagent_id) for t in child_turns[subagent.id]])
        if (not child_turns[subagent.id] or any(
            t.status != "completed" or t.subagent_id != subagent.id for t in child_turns[subagent.id]
        )):
            raise RuntimeError("Worker turns are missing, incomplete, failed, cancelled, or owned by another worker")
    root_items = list(client.beta.agents.sessions.items.list(result.session_id, order="asc", limit=100, timeout=15))
    history_checked = True


### 3. 역할별 스크립트와 실행 주체 확인

업무 역할은 생성 순서 대신 실제 실행한 스크립트 경로로 연결합니다. 아래는 이번 실습이 요청한 별도 python 스크립트 명령을 확인합니다. 명령을 단순 언급한 echo·코드 작성 명령은 실행으로 표시하지 않습니다. 임의 shell 래퍼나 복잡한 파이프라인을 분석하는 일반 감사기는 아닙니다.

각 명령의 하위 Turn을 직접 조회해 ID·소유자·완료 상태를 확인하고 두 역할의 담당이 다른지 검사합니다. 메인의 성공한 Python 명령도 표시합니다. 메인이 두 파일을 실제로 통합했는지와 계산 정답은 별도 검증 사본에서 대조합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    commands_checked = False
    if not history_checked:
        raise RuntimeError("Retrieve worker histories in step 2 first")
    command_actors, role_executions = {}, {}
    print("[3/6] Role | Subagent | Turn | Command | Exit code")
    for script in ("structure.py", "target.py"):
        candidates = []
        for sid, items in child_items.items():
            for item in items:
                if item.type != "command_execution" or item.status != "completed" or item.exit_code != 0:
                    continue
                try:
                    tokens = shlex.split(item.command)
                except ValueError:
                    continue
                if len(tokens) == 2 and Path(tokens[0]).name in {"python", "python3"} and tokens[1] == "/workspace/workers/" + script:
                    candidates.append((sid, item))
        if len({sid for sid, _ in candidates}) != 1:
            raise RuntimeError("Missing or ambiguous successful worker execution: " + script)
        sid, command = candidates[-1]
        command_turn = client.beta.agents.sessions.subagents.turns.retrieve(
            command.turn_id, session_id=result.session_id, subagent_id=sid, timeout=15,
        )
        if command_turn.id != command.turn_id or command_turn.subagent_id != sid or command_turn.status != "completed":
            raise RuntimeError("Command turn identity, owner, or completion differs")
        command_actors[sid] = command_turn.subagent_id
        role_executions[script] = {"subagent_id": sid, "turn_id": command.turn_id, "command_item_id": command.id}
        print(f"{script} | {sid} | {command.turn_id} | {command.command} | {command.exit_code}")
    if len({entry["subagent_id"] for entry in role_executions.values()}) != 2:
        raise RuntimeError("The two roles must run in different workers")
    for item in root_items:
        if item.type == "command_execution" and item.turn_id == turn.id and item.status == "completed" and item.exit_code == 0:
            try:
                tokens = shlex.split(item.command)
            except ValueError:
                continue
            if tokens and Path(tokens[0]).name in {"python", "python3"}:
                print("Coordinator command:", item.id, item.turn_id, item.command, item.exit_code)
    print("Final result:", result.output_text)
    commands_checked = True


### 4. 공유 환경에서 네 결과 파일 다운로드

Artifacts는 파일 게시 기록입니다. 하위 담당 파일은 다른 Turn에서 게시될 수 있으므로 메인 Turn만으로 제한하지 않습니다. 이번 실행에서 조회한 완료 Turn들의 집합과 정확한 경로로 선택하고, 누락·중복은 파일명을 포함한 오류로 표시합니다. 게시 Turn이 파일 작성자를 단독 증명하는 것은 아니므로 3단계 명령 이력과 함께 읽으세요.

각 파일의 Artifact ID·게시 Turn·원격/로컬 경로를 확인합니다. 네 파일을 모두 받은 뒤에만 report_downloaded가 True가 됩니다. 다운로드에 실패하면 오류 정리 정책을 적용하며 로컬 일부 파일만으로 성공을 판단하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    report_downloaded = False
    downloaded_artifacts = []
    if not commands_checked:
        raise RuntimeError("Inspect the role commands in step 3 first")
    print("[4/6] Downloading the four published results...")
    published = list(client.beta.agents.sessions.artifacts.list(result.session_id, limit=100, timeout=15))
    completed_turn_ids = {turn.id} | {t.id for turns in child_turns.values() for t in turns if t.status == "completed"}
    downloads = output_dir / "downloads"
    downloads.mkdir(exist_ok=True)
    for name in ("sales_breakdown.json", "target_assessment.json", "summary.json", "report.md"):
        path = "/workspace/outputs/" + name
        matches = [a for a in published if a.path == path and a.turn_id in completed_turn_ids]
        if len(matches) != 1:
            raise RuntimeError(f"Expected one artifact for {name}; found {len(matches)}")
        artifact = matches[0]
        with client.beta.agents.sessions.artifacts.with_streaming_response.content(
            artifact.id, session_id=result.session_id, timeout=30,
        ) as content:
            content.stream_to_file(downloads / name)
        downloaded_artifacts.append({"id": artifact.id, "turn_id": artifact.turn_id, "path": artifact.path})
        print("Artifact:", downloaded_artifacts[-1], "Local path:", downloads / name)
    report_downloaded = True
    print("All four files downloaded:", report_downloaded)


### 5. 담당 결과와 통합 보고서 읽기

sales_breakdown.json의 metrics·by_date, target_assessment.json의 기간·목표·달성률·부족액을 먼저 읽습니다. 두 담당의 총매출과 summary.json의 총매출이 일치하는지 확인한 뒤 report.md를 표시합니다. 이는 담당 결과 간 일관성 확인이며 CSV 정답을 계산해 비교하는 자동 검증과 다릅니다.

로컬 파일은 Session 삭제 후에도 남습니다. 다만 이 단계에서는 아직 삭제하지 않았으므로, 결과 확인 뒤 6단계 정리를 실행하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not report_downloaded:
        raise RuntimeError("Download all four files in step 4 first")
    print("[5/6] Reading worker results before the integrated report...")
    structure = json.loads((downloads / "sales_breakdown.json").read_text())
    assessment = json.loads((downloads / "target_assessment.json").read_text())
    summary = json.loads((downloads / "summary.json").read_text())
    print("Sales structure:", structure)
    print("Target assessment:", assessment)
    print("Integrated summary:", summary)
    revenues = (structure["metrics"]["total_revenue"], assessment["total_revenue"], summary["total_revenue"])
    if len(set(revenues)) != 1:
        raise RuntimeError("Worker and integrated revenues disagree; inspect the saved files")
    print("Revenue agreement across the three results:", revenues[0])
    display(Markdown((downloads / "report.md").read_text()))


### 6. 자원 정리와 최종 요약

중간에 학습을 멈춰도 이 셀은 실행하세요. 실행 중인 작업은 정리 helper가 취소를 시도한 뒤 Session 삭제·부재를 확인합니다. 정리하면 client도 닫히므로 기존 client로 조회를 이어가지 않습니다. 원래 실행·검증 오류가 있으면 정리 오류로 덮지 않고 추가 정보로 남깁니다. 다운로드한 파일은 로컬에 남습니다.


In [ ]:
try:
    print("[6/6] Cleaning up the session and shared environment...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({
    "root_status": turn.status if turn is not None else "not_completed",
    "observed_subagents": len(created_subagent_ids),
    "roles_checked": commands_checked,
    "all_files_downloaded": report_downloaded,
    "session_cleanup": "verified" if sessions else "not_needed",
})


## 완료 기준과 실패 처리

서로 다른 두 담당의 완료 Turn·역할별 성공한 Python 명령, 메인의 통합 명령, 담당 JSON과 통합 보고서를 함께 확인합니다. 생성 이벤트와 저장 목록을 대조하고, 파일 내용만 맞거나 모델이 완료했다고 말한 경우를 역할 분담 성공으로 단정하지 않습니다. 담당 JSON의 작성자 문자열도 증거로 사용하지 않습니다.

| 상황 | 다음 행동 |
| --- | --- |
| 담당 Turn 실패·취소 또는 결과 누락 | 성공 통합으로 처리하지 않고 해당 이력 확인, 정리 상태 점검 |
| 파일 누락·중복 또는 다운로드 실패 | 임의로 결과를 채우지 않고 해당 경로·게시 Turn 확인 |
| 스트림 중단·timeout | 원래 Session의 저장 상태부터 조사, 같은 작업 무조건 재전송 금지 |
| 중간에 학습 중단 | 6단계 정리 실행, 강제 종료라면 creation.json의 본 실습 자원 확인 |

오류 뒤 helper가 자원을 정리했을 수 있으므로 닫힌 client로 복구 조회를 이어가지 않습니다. [14장 관찰 연결 복구](14_operations.ipynb)는 같은 Session 상태 조회와 재전송 방지를 설명합니다. 이 장은 자동 재개·장애를 넘는 exactly-once를 구현하지 않습니다.

수동 실행은 단계별 출력으로 확인합니다. 검증 스크립트만 실행 사본에 정답 비교·세션 삭제 전 이력 검사·실패/정리 receipt를 추가합니다. 강제 종료 시 receipt와 정리는 보장하지 않습니다.

로컬 실행 명령: uv run --group dev python scripts/run_multi_agent.py --live. API 없이 준비 셀만 검증하려면 --live를 생략하세요. 실행 사본은 outputs/extensions-validation/15_multi_agent.executed.ipynb, 다운로드·검증 기록은 outputs/extensions/15_multi_agent/ 아래에 저장합니다.


## 선택 확장과 관찰 범위

- 9장 Skill 또는 8장 MCP로 분석 기능·목표 조회를 확장할 수 있습니다. 기본 실습은 JSON 파일만 사용합니다.
- 하위 에이전트는 MCP·환경의 명령·파일을 사용할 수 있으나 공식 문서상 Function Tool은 지원하지 않습니다. 7장의 함수 핸들러를 하위 에이전트에 그대로 적용하지 않습니다.
- 11장의 검증·승인은 메인이 통합한 보고서에 적용합니다.
- 더 깊은 실행 조사는 13장의 기본 Trace 확인 경로를 사용합니다. 스트리밍 항목이 에이전트 간 전체 대화 기록을 제공한다고 가정하지 않습니다.

기본 실습은 목표 JSON 파일과 직접 작성한 Python 스크립트를 사용합니다. MCP·Skill·승인 조합은 선택 확장 아이디어이며 실제 실행 검증 범위에 포함되지 않습니다. API·Docker·Trace export의 실제 실행 범위를 구분해 기록하며 본 장은 Trace export 없이 실행할 수 있습니다.

## 공식 참고 자료

### Agents API 전용 문서

- [Multi-agent](https://developers.openai.com/api/docs/guides/agents-api/multi-agent) — 설정·공유 환경·담당별 실행 이력.
- [Files and artifacts](https://developers.openai.com/api/docs/guides/agents-api/environments/files) — 입력 배치·결과 다운로드.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 스트리밍·저장 결과·복구.
- [Observability](https://developers.openai.com/api/docs/guides/agents-api/observability) — root와 하위 에이전트 실행 관찰.

실행 방식과 검증 범위는 [실습 안내](../../docs/validation.md)를 참고하세요. 오프라인 검사와 실제 API 실행 검증은 구분합니다.
